# Exercise 1 — HTTP & REST

**After:** [Day 1](../lessons/day1-http-and-rest/01_theory_http_and_rest.md)

Three levels. Try each before opening the solution — being stuck for two minutes is where the
learning happens.

In [ ]:
import json

from fastapi import FastAPI, HTTPException, status
from fastapi.testclient import TestClient

def show(r, label=""):
    print(f"{label:<40} {r.status_code}  {r.text[:120]}")

print("ready")

## ⭐ Level 1 — Classify

For each situation, write the **method**, the **path**, and the **status code** you'd expect on
success. Then check yourself.

1. Fetch every station
2. Fetch station `260`
3. Add a new station
4. Change only station 260's name
5. Remove station 260
6. Fetch stations in the province of Utrecht only

In [ ]:
# Your answers here, e.g.:
# 1. GET /stations -> 200


<details>
<summary>💡 Solution</summary>

| # | Method | Path | Success code |
|---|---|---|---|
| 1 | `GET` | `/stations` | `200` |
| 2 | `GET` | `/stations/260` | `200` |
| 3 | `POST` | `/stations` | `201` |
| 4 | `PATCH` | `/stations/260` | `200` |
| 5 | `DELETE` | `/stations/260` | `204` |
| 6 | `GET` | `/stations?province=Utrecht` | `200` |

Number 6 is the one people get wrong: **province is a filter, not an identity.** `/provinces/Utrecht/stations`
is also defensible (a sub-collection), but `/stations/Utrecht` is not — that path says "the station
called Utrecht".

Number 4: `PATCH` for a partial change, `PUT` if you're replacing the whole record. `PUT` is
idempotent; `PATCH` generally is not.
</details>

## ⭐⭐ Level 2 — Build it and prove the codes

Build a small station API with `GET /stations`, `GET /stations/{code}` (404 when unknown),
`POST /stations` (201), and `DELETE /stations/{code}` (204). Then write a loop that prints the status
code of each call, including two failures.

In [ ]:
# Your code here


<details>
<summary>💡 Solution</summary>

```python
app = FastAPI()
STATIONS = {"260": {"code": "260", "name": "De Bilt"},
            "240": {"code": "240", "name": "Schiphol"}}

@app.get("/stations")
def list_stations(province: str | None = None):
    return list(STATIONS.values())

@app.get("/stations/{code}")
def read_station(code: str):
    if code not in STATIONS:
        raise HTTPException(404, detail=f"No station {code!r}")
    return STATIONS[code]

@app.post("/stations", status_code=status.HTTP_201_CREATED)
def create_station(code: str, name: str):
    if code in STATIONS:
        raise HTTPException(409, detail=f"Station {code!r} already exists")
    STATIONS[code] = {"code": code, "name": name}
    return STATIONS[code]

@app.delete("/stations/{code}", status_code=status.HTTP_204_NO_CONTENT)
def delete_station(code: str):
    STATIONS.pop(code, None)      # pop with a default: deleting twice is fine
    return None

c = TestClient(app)
for method, path in [
    ("GET", "/stations"), ("GET", "/stations/260"), ("GET", "/stations/999"),
    ("POST", "/stations?code=344&name=Rotterdam"),
    ("POST", "/stations?code=344&name=Rotterdam"),      # again -> 409
    ("DELETE", "/stations/344"), ("DELETE", "/stations/344"),   # again -> still 204
]:
    show(c.request(method, path), f"{method} {path}")
```

Two things to notice. `POST` twice gives **409 Conflict** here because we chose to detect the
duplicate — without that check you'd silently overwrite. `DELETE` twice returns `204` both times,
because `DELETE` is **idempotent**: the desired end state ("station 344 is gone") is achieved either
way, so the second call is not an error.
</details>

## ⭐⭐⭐ Level 3 — Design under constraints

Design the endpoints for a service that lets a user:

1. Ask for a **forecast** for a city (read-only, cacheable)
2. **Subscribe** to a daily email alert for a city above a temperature threshold
3. List their subscriptions
4. Cancel one
5. Trigger a **manual refresh** of the forecast cache

For each: method, path, where the parameters live, and the success code. Number 5 is the
interesting one — justify your choice.

In [ ]:
# Your answers here


<details>
<summary>💡 Solution</summary>

| # | Design | Notes |
|---|---|---|
| 1 | `GET /forecasts?city=Utrecht&days=7` → `200` | Read-only, so `GET`; city and horizon are options |
| 2 | `POST /subscriptions` body `{"city": "Utrecht", "threshold_c": 25}` → `201` | Creating a resource; the details are a body, not a query string |
| 3 | `GET /subscriptions` → `200` | Scoped to the authenticated caller |
| 4 | `DELETE /subscriptions/{id}` → `204` | Identity in the path |
| 5 | `POST /refreshes` → `202 Accepted`, returning `{"id": 17, "status": "running"}` | See below |

**Why number 5 is interesting.** "Do a thing" doesn't fit nouns-and-verbs neatly, and the tempting
answer is `POST /refresh` or, worse, `GET /triggerRefresh`.

The professional move is to **make the action a resource**: a refresh is a thing that gets created,
has an id, and can be inspected later with `GET /refreshes/17`. That turns a fire-and-forget call
into something the caller can poll — which matters, because a cache refresh takes longer than a
request should.

`202 Accepted` is the right code: *"I've taken your request, it isn't finished."* Not `200` (which
would imply the work is done) and not `201` unless you're genuinely returning the finished resource.
</details>